<a href="https://colab.research.google.com/github/Makasane/ECommerce-Retail-Analytics-Audi/blob/main/Notebooks/store_schema_lifecycle_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd

print(" [MASTER AUDIT] Scanning All Uploaded Data Assets in Workspace Layer...\n")

# Dynamically locate every single CSV file inside your Google Colab folder panel
all_uploaded_files = [f for f in os.listdir('.') if f.endswith('.csv')]

if len(all_uploaded_files) == 0:
    print(" No CSV files detected. Make sure your files are uploaded to the left folder tab panel.")
else:
    print(f" Found {len(all_uploaded_files)} files to audit. Initializing schemas...\n")

for filename in all_uploaded_files:
    print("=" * 75)
    print(f" FILE DIAGNOSTIC: {filename}")
    print("=" * 75)

    try:
        # Optimized memory configuration to handle massive files like amazon_products.csv safely
        # We read just the top 5 rows to extract headers and data type rules instantly without crashing
        df_preview = pd.read_csv(filename, nrows=5)

        # Calculate true record volume count without loading millions of rows into RAM
        with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
            true_total_rows = sum(1 for line in f) - 1

        print(f" Volumetric Row Count   : {true_total_rows:,} total entries")
        print(f" Dimensional Column Count: {len(df_preview.columns)}")

        # Display the column headers and structural mappings
        print("\n Column Schema & Headings Matrix:")
        schema_df = pd.DataFrame({
            'Data Type Sample': df_preview.dtypes.astype(str),
            'First Row Value': df_preview.iloc[0].astype(str)
        })
        print(schema_df)

    except Exception as e:
        print(f" Could not audit file {filename} due to system constraint: {e}")

    print("\n" + "-"*75 + "\n")


 [MASTER AUDIT] Scanning All Uploaded Data Assets in Workspace Layer...

 Found 6 files to audit. Initializing schemas...

 FILE DIAGNOSTIC: amazon_categories.csv
 Volumetric Row Count   : 248 total entries
 Dimensional Column Count: 2

 Column Schema & Headings Matrix:
              Data Type Sample           First Row Value
id                       int64                         1
category_name           object  Beading & Jewelry Making

---------------------------------------------------------------------------

 FILE DIAGNOSTIC: certified_compressed_retail_output.csv
 Volumetric Row Count   : 20,876 total entries
 Dimensional Column Count: 17

 Column Schema & Headings Matrix:
                   Data Type Sample First Row Value
transaction_id                int64     29258453508
cust_id                       int64          270384
tran_date                   float64             nan
prod_subcat_code              int64               5
prod_cat_code                 int64               3

In [7]:
import pandas as pd
import numpy as np

print(" [PIPELINE INITIALIZED] Re-generating and Analyzing Integrated Dataset...")

# 1. Ingest base tables directly from your left panel
try:
    products = pd.read_csv('amazon_products.csv')
    categories = pd.read_csv('amazon_categories.csv')
except FileNotFoundError as e:
    print(f" Error: Could not locate files in panel. Please check filenames. Details: {e}")

# 2. Apply cleaning and boundary conditions
products['listPrice'] = np.where(products['listPrice'] == 0, products['price'], products['listPrice'])
products['listPrice'] = products['listPrice'].fillna(products['price'])
products_optimized = products.drop(columns=['imgUrl', 'productURL'], errors='ignore').copy()

# 3. Perform Relational Star Schema Join
amazon_master_layer = pd.merge(
    products_optimized,
    categories,
    left_on='category_id',
    right_on='id',
    how='left'
)
amazon_master_layer = amazon_master_layer.drop(columns=['id'], errors='ignore')

# 4. Feature Engineer Corporate KPIs
amazon_master_layer['estimated_monthly_revenue'] = amazon_master_layer['price'] * amazon_master_layer['boughtInLastMonth']
amazon_master_layer['markdown_discount_percent'] = ((amazon_master_layer['listPrice'] - amazon_master_layer['price']) / amazon_master_layer['listPrice']) * 100
amazon_master_layer['markdown_discount_percent'] = amazon_master_layer['markdown_discount_percent'].fillna(0)

# Save a local instance to make sure it exists
amazon_master_layer.to_csv('certified_amazon_analytics_output.csv', index=False)

# 5. Extract Master Validation Control Figures
total_revenue = amazon_master_layer['estimated_monthly_revenue'].sum()
total_records = len(amazon_master_layer)
total_reviews = amazon_master_layer['reviews'].sum()

top_categories = amazon_master_layer.groupby('category_name')['estimated_monthly_revenue'].sum().reset_index()
top_categories = top_categories.sort_values(by='estimated_monthly_revenue', ascending=False).head(3)

print("=" * 70)
print(" MASTER BALANCES (Write these down to verify your Looker Dashboard!)")
print("=" * 70)
print(f" TARGET GRAND TOTAL REVENUE  : ${total_revenue:,.2f}")
print(f" TARGET TOTAL RECORD COUNT   : {total_records:,} rows")
print(f" TARGET TOTAL REVIEWS COUNT  : {total_reviews:,} reviews")
print("-" * 70)
print("\n TOP 3 DEPARTMENTS TO CROSS-CHECK:")
for idx, row in top_categories.iterrows():
    print(f"  • {row['category_name']}: ${row['estimated_monthly_revenue']:,.2f}")
print("=" * 70)


 [PIPELINE INITIALIZED] Re-generating and Analyzing Integrated Dataset...


/tmp/ipykernel_19972/828077610.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  products = pd.read_csv('amazon_products.csv')


 MASTER BALANCES (Write these down to verify your Looker Dashboard!)
 TARGET GRAND TOTAL REVENUE  : $475,856,119.00
 TARGET TOTAL RECORD COUNT   : 161,424 rows
 TARGET TOTAL REVIEWS COUNT  : 103,647,362.0 reviews
----------------------------------------------------------------------

 TOP 3 DEPARTMENTS TO CROSS-CHECK:
  • Office Electronics: $95,038,114.00
  • Vacuum Cleaners & Floor Care: $81,262,567.50
  • Toys & Games: $67,796,243.50


In [9]:
import pandas as pd

# Load the brick-and-mortar output file
df = pd.read_csv('certified_compressed_retail_output.csv')

print(" [RETAIL STORE ENGINE] Extracting Master Validation Numbers...\n")

# Calculate Grand Totals for Project 2
total_revenue = df['total_amt'].sum()
total_records = len(df)
avg_tax = df['Tax'].mean()

print("=" * 70)
print(" STORE BALANCES (Write these down to match your current Looker screen!)")
print("=" * 70)
print(f" TARGET GRAND TOTAL REVENUE  : R{total_revenue:,.2f}")
print(f" TARGET TOTAL RECORD COUNT   : {total_records:,} rows")
print(f" TARGET AVERAGE TAX MARGIN   : R{avg_tax:,.2f}")
print("=" * 70)


 [RETAIL STORE ENGINE] Extracting Master Validation Numbers...

 STORE BALANCES (Write these down to match your current Looker screen!)
 TARGET GRAND TOTAL REVENUE  : R54,453,885.07
 TARGET TOTAL RECORD COUNT   : 20,876 rows
 TARGET AVERAGE TAX MARGIN   : R247.86


In [10]:
import pandas as pd
import numpy as np

# Load our primary store schema dataset
df = pd.read_csv('certified_compressed_retail_output.csv')

print(" [ADVANCED AUDIT VALIDATION] Running Structural Data Metric Scans...\n")

# 1. Channel Revenue Share Breakdown (Interpretation Support)
channel_revenue = df.groupby('Store_type')['total_amt'].sum().reset_index()
channel_revenue['Revenue_Share_Percent'] = (channel_revenue['total_amt'] / df['total_amt'].sum()) * 100

print(" CHANNEL PERFORMANCE DISTRIBUTION:")
for idx, row in channel_revenue.sort_values(by='total_amt', ascending=False).iterrows():
    print(f"  • {row['Store_type']}: R{row['total_amt']:,.2f} ({row['Revenue_Share_Percent']:.2f}%)")

# 2. Tax Leakage & Integrity Assessment
# Standard retail tax checking: Let's calculate the explicit effective tax rate across transactions
df['effective_tax_rate'] = (df['Tax'] / (df['Qty'] * df['Rate'])) * 100
mean_tax_rate = df['effective_tax_rate'].mean()

print("\n TAX INFRASTRUCTURE INTEGRITY:")
print(f"  • Mean Calculated System Effective Tax Rate: {mean_tax_rate:.2f}%")
# If this number rests uniformly around a standard value, our tax engine logic is verified.

# 3. Data Outlier Variance Audit
# Spotting extreme high-value bulk purchasing lines that could skew dashboard averages
revenue_baseline = df['total_amt']
q1 = revenue_baseline.quantile(0.25)
q3 = revenue_baseline.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + (1.5 * iqr)

outlier_transactions = df[df['total_amt'] > upper_bound]
print(f"\n TRANSACTONAL ANOMALY DETECTION:")
print(f"  • Isolated {len(outlier_transactions)} extreme value outlier transactions above boundary threshold: R{upper_bound:,.2f}")
print(f"  • Outlier Data Contribution: R{outlier_transactions['total_amt'].sum():,.2f} of total asset volume.")


 [ADVANCED AUDIT VALIDATION] Running Structural Data Metric Scans...

 CHANNEL PERFORMANCE DISTRIBUTION:
  • e-Shop: R22,185,609.88 (40.74%)
  • MBR: R10,908,536.79 (20.03%)
  • Flagship store: R10,906,619.62 (20.03%)
  • TeleShop: R10,453,118.78 (19.20%)

 TAX INFRASTRUCTURE INTEGRITY:
  • Mean Calculated System Effective Tax Rate: 10.50%

 TRANSACTONAL ANOMALY DETECTION:
  • Isolated 152 extreme value outlier transactions above boundary threshold: R8,017.74
  • Outlier Data Contribution: R1,240,406.70 of total asset volume.


In [11]:
import pandas as pd
import numpy as np

# Load our primary store schema dataset
df = pd.read_csv('certified_compressed_retail_output.csv')

print("🔒 [SYSTEMIC VARIANCE LOG] Generating Data Integrity Validation Audit...\n")

# 1. Systemic Tax Variance Check
df['calculated_tax_rate'] = (df['Tax'] / (df['Qty'] * df['Rate'])) * 100
tax_variance = df['calculated_tax_rate'].std()

print("=" * 70)
print(" DATA INTEGRITY METRICS LOG")
print("=" * 70)
print(f" SYSTEMIC TAX CONSTANT      : {df['calculated_tax_rate'].mean():.2f}%")
print(f" TAX VARIANCE COEFFICIENT   : {tax_variance:.6f} (Flawless Stability)")

# 2. Check for Duplication Anomalies across Channels
mbr_total = df[df['Store_type'] == 'MBR']['total_amt'].sum()
flag_total = df[df['Store_type'] == 'Flagship store']['total_amt'].sum()
variance_gap = abs(mbr_total - flag_total)

print(f" CHANNEL VARIANCE GAP (MBR vs Flagship): R{variance_gap:,.2f}")
if variance_gap == 0:
    print("   ALERT: MBR and Flagship store exhibit a 100% statistical mirror. Potential data symmetry constraint.")

# 3. Structural Record Validation Bounds
print("-" * 70)
print(" RECONCILIATION SUMMARY FOR RECRUITERS:")
print(f"  • Total Validated Cash Volume : R{df['total_amt'].sum():,.2f}")
print(f"  • Active Transaction Volume  : {len(df):,} records")
print(f"  • Average Transaction Size   : R{df['total_amt'].mean():,.2f}")
print("=" * 70)


 [SYSTEMIC VARIANCE LOG] Generating Data Integrity Validation Audit...

 DATA INTEGRITY METRICS LOG
 SYSTEMIC TAX CONSTANT      : 10.50%
 TAX VARIANCE COEFFICIENT   : 0.000000 (Flawless Stability)
 CHANNEL VARIANCE GAP (MBR vs Flagship): R1,917.18
----------------------------------------------------------------------
 RECONCILIATION SUMMARY FOR RECRUITERS:
  • Total Validated Cash Volume : R54,453,885.07
  • Active Transaction Volume  : 20,876 records
  • Average Transaction Size   : R2,608.44
